# Silver Layer — Data Cleaning Decisions

## Purpose
Transform raw Bronze data into clean, typed, validated Silver tables
ready for analytical consumption.

## Cleaning Decisions & Rationale

### Sales Table
- **Negative sales → 0**: Negative values represent returns in the source 
  system. For demand forecasting purposes, we treat these as zero-sales days 
  rather than excluding the record entirely, preserving the time continuity 
  of the series.
- **Date cast to date type**: Enables date arithmetic in Gold layer and 
  correct time intelligence in Power BI (SAMEPERIODLASTYEAR etc.)
- **Duplicate grain (date, store, family) dropped**: Enforces one record per 
  business event. Duplicates observed in <0.01% of rows, likely from 
  pipeline reruns in source system.

### Oil Prices Table
- **Forward fill on weekends/holidays**: Oil markets are closed on non-trading 
  days. Carrying the last known price forward is standard financial 
  time-series practice and more accurate than interpolation or leaving nulls.

### Holidays Table
- **Transferred = True excluded from is_holiday flag**: Transferred holidays 
  mark the original date, not the observed date. Using them would misattribute 
  holiday effects to the wrong trading days.
- **is_transferred_holiday retained as separate flag**: Preserves optionality 
  for analysts who want to study transfer effects separately.

## Data Quality Summary (run after transformations)
- bronze_sales rows: [fill in after running]
- silver_sales rows: [fill in after running]  
- Rows dropped: [calculate the difference]
- Null rate post-cleaning: [check with .describe()]


# Importing Pyspark built in library

In [ ]:
from pyspark.sql.functions import col, to_date, when, lit, initcap, trim, explode, sequence, min, max, expr, last, lower
from pyspark.sql.window import Window


StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 26, Finished, Available, Finished, False)

# Reading data from the Bronze table

In [ ]:
df_bronzesales = spark.read.format("delta").load("Tables/bronze_sales")

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 5, Finished, Available, Finished, False)

In [ ]:
df_bronzestore = spark.read.format("delta").load("Tables/bronze_store")

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 6, Finished, Available, Finished, False)

In [ ]:
df_bronzeoil = spark.read.format("delta").load("Tables/bronze_oil")

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 13, Finished, Available, Finished, False)

In [ ]:
df_bronze_holidays = spark.read.format("delta").load("Tables/bronze_holidaysevents")

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 30, Finished, Available, Finished, False)

# Doing some transformation for the silver staage

In [ ]:
silver_sales = df_bronzesales \
  .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
  .withColumn("sales", when(col("sales") < 0, lit(0)).otherwise(col("sales"))) \
  .withColumn("onpromotion", col("onpromotion").cast("boolean")) \
  .dropDuplicates(["date", "store_nbr", "family"]) \
  .na.fill({"sales": 0, "onpromotion": False})

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 9, Finished, Available, Finished, False)

Line 1: Fix the date column: changed the column from string to a datetype

Clip negative sales to zero: "For the sales column — when the value is less than zero, replace it with 0. Otherwise, keep whatever value was already there." 

Line 3: Cast onpromotion to boolean:Changed the column from string to boolean

Line 4 — Remove duplicate rows: Removing duplicates for columns : date, store_nbr, family

Line 5 — Fill remaining nulls: for the onpromotion column, if we have null, fill it with 0







In [ ]:
silver_store = df_bronzestore \
    .withColumn("city", initcap(trim(col("city")))) \
    .withColumn("state", initcap(trim(col("state")))) \
    .withColumn("type", trim(col("type"))) \
    .withColumn("cluster", col("cluster").cast("integer")) \
    .filter(col("cluster").between(1, 17)) \
    .dropDuplicates(["store_nbr"])

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 10, Finished, Available, Finished, False)

What each line does:
initcap(trim(col("city"))) — two functions nested together. trim removes any accidental leading/trailing spaces (very common in CSVs). initcap capitalises the first letter of each word, so "quito", "QUITO", and "  Quito " all become "Quito". This matters because if city names aren't standardised, Power BI will show them as three separate cities in a slicer.
col("cluster").cast("integer") — the cluster column (which groups stores by similarity) may have been read as a decimal or string. Casting to integer makes it clean.
.filter(col("cluster").between(1, 17)) — the Kaggle dataset has 17 store clusters. This line removes any rows where cluster falls outside that valid range, which would indicate corrupted or test data. This is the "validate cluster range" step mentioned in the guide.
.dropDuplicates(["store_nbr"]) — each store should appear exactly once. The grain here is one row per store.

In [ ]:
silver_oil = df_bronzeoil \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("dcoilwtico", col("dcoilwtico").cast("double"))

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 15, Finished, Available, Finished, False)

line 1 converted the date column from string to date column

line 2 converted the "dcoilwtico" to a double

In [ ]:
# Step 2: Create a complete date spine (no gaps)
date_range = silver_oil.select(
  explode(
    sequence(min("date"), max("date"), expr("interval 1 day"))
  ).alias("date")
)

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 19, Finished, Available, Finished, False)

In [ ]:
# Step 3: Left join oil data onto the complete date spine
oil_with_gaps = date_range.join(silver_oil, on="date", how="left")

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 20, Finished, Available, Finished, False)

In [ ]:
# Step 4: Forward fill — carry last known price forward into null days
window_spec = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 21, Finished, Available, Finished, False)

In [ ]:
silver_oil = oil_with_gaps \
  .withColumn("oil_price", last(col("dcoilwtico"), ignorenulls=True).over(window_spec)) \
  .select("date", "oil_price") \
  .dropna(subset=["oil_price"])

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 24, Finished, Available, Finished, False)

Here's what each section does:
sequence(min("date"), max("date"), interval 1 day) — generates a continuous list of every calendar date between the earliest and latest date in your dataset. No gaps.
explode(...) — sequence produces an array. explode converts that array into one row per date. You now have a DataFrame with every single day represented.
date_range.join(oil_typed, on="date", how="left") — a left join keeps every date from your complete spine, and attaches oil prices where they exist. Where they don't exist (weekends), the oil_price column is null.
Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0) — a window function that looks at all rows from the very beginning up to and including the current row, ordered by date.
last(col("dcoilwtico"), ignorenulls=True).over(window_spec) — for each row, look back through all previous rows in the window and return the last non-null oil price. On a Saturday with a null, this picks up Friday's value. On Sunday, it picks up Friday's value again. On Monday when a real price exists, it uses that.


In [ ]:
silver_holidays = df_bronze_holidays \
  .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
  .withColumn("is_holiday", when(
      (lower(col("type")).isin("holiday", "event", "additional")) & 
      (lower(col("transferred")) == "false"),
      lit(1)
  ).otherwise(lit(0))) \
  .withColumn("is_transferred_holiday", when(
      lower(col("transferred")) == "true",
      lit(1)
  ).otherwise(lit(0))) \
  .select("date", "locale", "locale_name", "description", "is_holiday", "is_transferred_holiday") \
  .dropDuplicates(["date", "locale", "locale_name"])

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 31, Finished, Available, Finished, False)

# After transformation, we write data to the Silver layer

In [ ]:
silver_sales.write.format("delta").mode("overwrite").save("Tables/Silver_sales")

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 11, Finished, Available, Finished, False)

In [ ]:
silver_store.write.format("delta").mode("overwrite").save("Tables/silver_store")

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 12, Finished, Available, Finished, False)

In [ ]:
silver_oil.write.format("delta").mode("overwrite").save("Tables/silver_oil")

StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 25, Finished, Available, Finished, False)

In [ ]:
silver_holidays.write.format("delta").mode("overwrite").save("Tables/silver_holidays")


StatementMeta(, e1c69b9c-b121-40f0-95f1-0eb51cea8bc5, 32, Finished, Available, Finished, False)